# 第8章　やってみる ― 代表的な領域から選ぶ簡単実装**『ゼロから動かす医療診断支援AI（入門編）』のコード**本文に載っているコードを、章の順にそのまま収めています。紙面のコードは読んで理解するためのもの、こちらは動かすためのものです。- Python 以外（シェル・YAML・Dockerfile など）は、実行環境が違うので**コードセルにせず、そのまま読める形で置いています**。使う場所を確かめてから実行してください。- 抜粋である以上、上から順に実行するだけで通るとは限りません。データの取得先やパスは、お手元の環境に合わせてください。- **教育・研究のためのコードです。患者データをこのノートブックに置かないでください。**リポジトリ: https://github.com/kewel-corp/book-intro

## 8.1　いちばん取り組みやすい一歩 ― OCT画像の分類

```textoct/├── train/│   ├── normal/     # 正常のOCT断面│   └── abnormal/   # 異常所見のあるOCT断面（黄斑浮腫、新生血管、ドルーゼンなど）└── val/            # 評価用（同じ構成）```

```text!pip install timmimport timm, torchfrom torchvision import datasets, transformsfrom torch.utils.data import DataLoader# 画像の前処理（サイズをそろえ、テンソルに変換）norm = transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])   # 事前学習と同じ統計にそろえるtf = transforms.Compose([transforms.Resize((224,224)), transforms.ToTensor(), norm])train_ds = datasets.ImageFolder("oct/train", transform=tf)      # フォルダ名が辞書順でラベルに（abnormal=0, normal=1）。陽性列を取り違えないよう class_to_idx を確認train_loader = DataLoader(train_ds, batch_size=16, shuffle=True)device = "cuda" if torch.cuda.is_available() else "cpu"model = timm.create_model("efficientnet_b0", pretrained=True, num_classes=2).to(device)opt = torch.optim.Adam(model.parameters(), lr=1e-4)lossf = torch.nn.CrossEntropyLoss()for epoch in range(5):                          # まずは5エポックだけ試す    model.train()    for images, labels in train_loader:        images, labels = images.to(device), labels.to(device)        opt.zero_grad()        loss = lossf(model(images), labels)        loss.backward(); opt.step()    print(f"epoch {epoch} done")```

## 整形外科 ― X線で手首骨折を「あり／なし」に分類する

```text!pip install timmimport timm, torch, numpy as npfrom torchvision import datasets, transformsfrom torch.utils.data import DataLoader# X線は1ch → 3chへ複製。反転・回転でデータ拡張するtrain_tf = transforms.Compose([    transforms.Grayscale(num_output_channels=3),   # 1ch→3ch    transforms.Resize((384, 384)),    transforms.RandomHorizontalFlip(),             # 左右反転    transforms.RandomRotation(10),                 # ±10度の回転    transforms.ToTensor(),    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),  # ImageNetの統計。省くと学習が進まない])train_ds = datasets.ImageFolder("wrist/train", transform=train_tf)  # 辞書順で fracture=0, normal=1（陽性＝骨折は index 0）train_loader = DataLoader(train_ds, batch_size=16, shuffle=True)# クラス不均衡に備え、少数クラス（骨折）を重視する重みを損失にかけるcounts = np.bincount([y for _, y in train_ds.samples])   # 各クラスの枚数w = torch.tensor(counts.sum() / counts, dtype=torch.float32)  # 少ないほど重いdevice = "cuda" if torch.cuda.is_available() else "cpu"model = timm.create_model("efficientnet_b0", pretrained=True, num_classes=2).to(device)opt   = torch.optim.Adam(model.parameters(), lr=1e-4)lossf = torch.nn.CrossEntropyLoss(weight=w.to(device))   # ← ここが不均衡対策for epoch in range(5):    model.train()    for x, y in train_loader:        x, y = x.to(device), y.to(device)        opt.zero_grad()        loss = lossf(model(x), y)        loss.backward(); opt.step()    print("epoch", epoch, "loss", round(loss.item(), 4))```

## 脳神経 ― 頭部CTスライスから「出血あり／なし」を分類する

```text!pip install pydicom timmimport pydicom, numpy as np, torchdef window(hu, center, width):                 # HU値を指定の窓で0〜1に正規化    lo, hi = center - width/2, center + width/2    return np.clip((hu - lo) / (hi - lo), 0, 1)def load_ct_3window(path):                      # DICOM1枚 → 3窓を3chに    d = pydicom.dcmread(path)    hu = d.pixel_array * d.RescaleSlope + d.RescaleIntercept   # 生値→HU値    brain    = window(hu,   40,   80)           # 脳実質ウィンドウ    subdural = window(hu,   80,  200)           # 硬膜下ウィンドウ（薄い出血）    bone     = window(hu,  600, 2800)           # 骨ウィンドウ（値は一例。施設・目的で変わる）    return np.stack([brain, subdural, bone], -1).astype(np.float32)  # (H,W,3)class CTSliceDS(torch.utils.data.Dataset):    def __init__(self, files, labels): self.files, self.labels = files, labels    def __len__(self): return len(self.files)    def __getitem__(self, i):        img = torch.from_numpy(load_ct_3window(self.files[i])).permute(2, 0, 1)  # (3,H,W)        img = torch.nn.functional.interpolate(img[None], size=(384, 384), mode="bilinear",                                              align_corners=False, antialias=True)[0]  # 縮小時は antialias=True（既定はFalseでエイリアシングする）        return img, self.labels[i]# files=DICOMパスの一覧, labels=各スライスの0/1  → あとは同じ学習ループ```

## 消化器 ― 腹部CTから肝臓を塗り分ける（2次元U-Netの最小例）

```text!pip install segmentation-models-pytorch nibabelimport nibabel as nib, numpy as np, torchimport segmentation_models_pytorch as smpdef load_slices(vol_path, seg_path):    vol = nib.load(vol_path).get_fdata()          # (第0軸, 第1軸, スライス方向) のHU値。軸の意味と向きはファイルのaffineで決まる    seg = nib.load(seg_path).get_fdata()    vol = np.clip(vol, -100, 200)                 # 腹部（軟部）ウィンドウ    vol = (vol + 100) / 300.0                     # 0〜1に正規化    mask = (seg > 0).astype(np.float32)           # ラベル>0（肝臓＋腫瘍）を前景に    return vol, mask                              # Z枚のスライスに分けて使うdef dice(pred, gt, eps=1e-6):                     # 塗りの重なりを採点（1に近いほど良い）    pred = (pred > 0.5).float()    inter = (pred * gt).sum()    return (2 * inter + eps) / (pred.sum() + gt.sum() + eps)device = "cuda" if torch.cuda.is_available() else "cpu"model = smp.Unet("resnet34", in_channels=1, classes=1,                 encoder_weights="imagenet").to(device)   # 1ch入力のU-Netopt = torch.optim.Adam(model.parameters(), lr=1e-4)bce = torch.nn.BCEWithLogitsLoss()for epoch in range(5):                            # x:(B,1,H,W) スライス, m:(B,1,H,W) マスク    model.train()    for x, m in train_loader:        x, m = x.to(device), m.to(device)        opt.zero_grad(); loss = bce(model(x), m)        loss.backward(); opt.step()model.eval()                                      # 評価はDice係数でscores, empty_gt_fp = [], 0with torch.no_grad():    for x, m in val_loader:        prob = torch.sigmoid(model(x.to(device))).cpu()        for pi, mi in zip(prob, m):               # バッチごとではなく1スライスずつ数える            if mi.sum() > 0:                scores.append(float(dice(pi, mi)))   # 正解に肝臓が写っているスライス            elif (pi > 0.5).sum() > 0:                empty_gt_fp += 1                     # 正解が空なのに塗った＝偽陽性# 正解も予測も空のスライスはDiceが1.0になる。平均に混ぜると数字が大きく過大になるので、# 前景のあるスライスだけで平均し、「何枚で測ったか」と偽陽性の枚数を必ず併記する。print("肝臓Dice =", round(float(np.mean(scores)), 3),      f"（前景あり{len(scores)}枚。正解が空なのに塗ったスライス {empty_gt_fp}枚）")```

## 3次元CTを分類する ― 「患者単位」で評価する

In [ ]:
import numpy as np, torchfrom sklearn.metrics import roc_auc_score# 前提：model は整形外科・脳神経の例と同じ2クラス分類モデル（直前の消化器のU-Netではない）。#       device は "cuda" or "cpu"。# test_loader は (画像, 正解, 患者ID) を返すよう作り替えたもの。# 患者IDは「文字列」で返すのが安全。数値で返すと DataLoader がテンソルにまとめるので、# 素のまま辞書のキーにすると同じ患者でも別キーになり、集約が効かない（スライス単位の成績が出る）。# 下の行で文字列に直しているのは、その取り違えを防ぐため。patient_prob, patient_true = {}, {}model.eval()with torch.no_grad():    for x, y, pid in test_loader:        prob = model(x.to(device)).softmax(1)[:, 1].cpu().numpy()  # 各スライスの陽性確率        for pi, p, t in zip(pid, prob, y.numpy()):            key = pi if isinstance(pi, str) else str(int(pi))   # テンソル/数値でも安全にまとめる            patient_prob[key] = max(patient_prob.get(key, 0.0), float(p))  # 患者内で最大            patient_true[key] = max(patient_true.get(key, 0),  int(t))     # 1枚でも陽性なら陽性ids  = list(patient_prob)auc  = roc_auc_score([patient_true[i] for i in ids],                     [patient_prob[i] for i in ids])print("患者単位AUC =", round(float(auc), 3))

## 眼底画像で「糖尿病網膜症あり／なし」を分類する ― 公開データをColabだけで完走

In [ ]:
import timm, torch, numpy as npfrom torchvision import datasets, transformsfrom torch.utils.data import DataLoader, Subsetfrom sklearn.model_selection import train_test_split# 眼底はカラー(3ch)。回転・明るさ・コントラストで拡張（検証・テストは拡張しない）train_tf = transforms.Compose([    transforms.Resize((320, 320)),    transforms.RandomRotation(10), transforms.ColorJitter(brightness=0.2, contrast=0.2),    transforms.RandomHorizontalFlip(),   # 眼底は左右反転しても「反対の眼」としてありうる（可否の原則は整形外科の例の注意を参照）    transforms.ToTensor(),    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])])   # ImageNetの統計eval_tf  = transforms.Compose([    transforms.Resize((320, 320)), transforms.ToTensor(),    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])])   # ImageNetの統計ROOT = "/content/dr"full = datasets.ImageFolder(f"{ROOT}/train")                    # ラベル: DR=0, No_DR=1pos  = full.class_to_idx["DR"]                                  # 陽性（網膜症あり）のインデックスlabels = [y for _, y in full.samples]tr_idx, va_idx = train_test_split(range(len(full)), test_size=0.15,                                  stratify=labels, random_state=0)  # 層化して検証を切り出すtrain_ds = Subset(datasets.ImageFolder(f"{ROOT}/train", transform=train_tf), tr_idx)val_ds   = Subset(datasets.ImageFolder(f"{ROOT}/train", transform=eval_tf),  va_idx)test_ds  = datasets.ImageFolder(f"{ROOT}/test", transform=eval_tf)          # 独立したテストtrain_loader = DataLoader(train_ds, batch_size=32, shuffle=True)val_loader   = DataLoader(val_ds,   batch_size=32)test_loader  = DataLoader(test_ds,  batch_size=32)

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"model = timm.create_model("efficientnet_b0", pretrained=True, num_classes=2).to(device)opt   = torch.optim.Adam(model.parameters(), lr=1e-4)counts = np.bincount([labels[i] for i in tr_idx])              # 学習側のクラス枚数w = torch.tensor(counts.sum() / counts, dtype=torch.float32).to(device)lossf = torch.nn.CrossEntropyLoss(weight=w)                    # 不均衡対策from sklearn.metrics import roc_auc_scorebest_auc = float("-inf")     # 0.0 で始めると、AUCが0のとき一度も更新されず「学習前の重み」を最良として返すbest_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}   # ループ前に安全な初期値（現在の重み）for epoch in range(6):    model.train()    for x, y in train_loader:        x, y = x.to(device), y.to(device)        opt.zero_grad(); loss = lossf(model(x), y)        loss.backward(); opt.step()    # --- 検証AUC ---    model.eval(); ys, ps = [], []    with torch.no_grad():        for x, y in val_loader:            p = model(x.to(device)).softmax(1)[:, pos].cpu().numpy()   # 陽性=網膜症ありの確率            ps += list(p); ys += list((y.numpy() == pos).astype(int))  # 陽性を1に統一    auc = roc_auc_score(ys, ps)    if np.isfinite(auc) and auc > best_auc:                    # 一番良い重みを控えておく（NaN は失敗扱い）        best_auc, best_state = auc, {k: v.cpu().clone() for k, v in model.state_dict().items()}    print(f"epoch {epoch}  val AUC = {auc:.3f}")assert np.isfinite(best_auc), "有効な検証が一度も終わっていない（学習前の重みを最良として使わない）"

In [ ]:
from sklearn.metrics import confusion_matriximport pandas as pdmodel.load_state_dict(best_state)                             # ベスト重みを復元して評価model.eval(); paths = [p for p, _ in test_ds.samples]probs, trues = [], []with torch.no_grad():    for x, y in test_loader:        probs += list(model(x.to(device)).softmax(1)[:, pos].cpu().numpy())  # 陽性=網膜症ありの確率        trues += list((y.numpy() == pos).astype(int))                        # 陽性を1に統一pred = (np.array(probs) >= 0.5).astype(int)tn, fp, fn, tp = confusion_matrix(trues, pred, labels=[0, 1]).ravel()print(f"テスト  感度={tp/(tp+fn):.3f}  特異度={tn/(tn+fp):.3f}  AUC={roc_auc_score(trues, probs):.3f}")pd.DataFrame({"path": paths, "prob": probs, "true": trues, "pred": pred}) \  .sort_values("prob", ascending=False).to_csv("/content/dr_pred.csv", index=False)from google.colab import files; files.download("/content/dr_pred.csv")